### Layout test using PP-DocLayout_plus-L

In [1]:

#gpu memory not suffiicient 
import os
os.environ["FLAGS_fraction_of_gpu_memory_to_use"] = "0.6"
os.environ["FLAGS_eager_delete_tensor_gb"] = "0.0"



In [20]:
from paddleocr import PaddleOCR, LayoutDetection, FormulaRecognition

layout_model = LayoutDetection(
    model_name="PP-DocLayout_plus-L",
    threshold={2: 0.2, 12: 0.2, 14: 0.2},  # text, header, footer
    device='gpu:0',
    layout_nms=True,
    layout_merge_bboxes_mode= "large",

)
formula_model = FormulaRecognition()
ocr_model = PaddleOCR(
    text_detection_model_name="PP-OCRv5_server_det",
    text_recognition_model_name="PP-OCRv5_server_rec",
    lang='en',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    device="gpu:0"
    
)

Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-DocLayout_plus-L`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-FormulaNet_plus-M`.
C:\Users\LOQ\AppData\Local\Temp\ipykernel_16028\1668134698.py:12: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  ocr_model = PaddleOCR(
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Mode

In [3]:
from pathlib import Path
import cv2

img_path = Path("data/page_1.png")
output  = Path("output/check")
output.mkdir(exist_ok= True, parents= True)
img = cv2.imread(str(img_path))
layout_results = layout_model.predict(img)

for res in layout_results:
    res.save_to_img(output)

There is not input file name as reference for name of saved result file. So the saved result file would be named with timestamp and random number: `1760883776_4854`.


In [3]:
import os
from pathlib import Path
import cv2
import time
import json

# def polygon_to_bbox(polygon):
#     # converts polygon to bbox coordinates
#     x_coords = [point[0] for point in polygon]
#     y_coords = [point[1] for point in polygon]
#     return [min(x_coords), min(y_coords), max(x_coords), max(y_coords)]

# def boxes_overlap(box1, box2, threshold=0.5):
 
#     # Convert to [x1, y1, x2, y2] format 
#     x1_1, y1_1, x2_1, y2_1 = box1[0], box1[1], box1[2], box1[3]
#     x1_2, y1_2, x2_2, y2_2 = box2[0], box2[1], box2[2], box2[3]
    
#     # intersection calculation
#     x1_i = max(x1_1, x1_2)
#     y1_i = max(y1_1, y1_2)
#     x2_i = min(x2_1, x2_2)
#     y2_i = min(y2_1, y2_2)
    
#     if x2_i < x1_i or y2_i < y1_i:
#         return False
    
#     # intersection calculateion
#     intersection_area = (x2_i - x1_i) * (y2_i - y1_i)
#     box1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
    
#     # Check if box1 is mostly inside box2
#     overlap_ratio = intersection_area / box1_area if box1_area > 0 else 0
    
#     return overlap_ratio > threshold


In [4]:
def run_image_pipeline_nc(path, layout_model, ocr):
    image_path = Path(path)
    now = time.time()
 
    # --- Layout analysis ---
    output_layout = layout_model.predict(str(image_path))
    time_after_layout = time.time()
    
    # --- OCR ---
    output_ocr = ocr.predict(str(image_path))
    time_after_ocr = time.time()
    
    # --- output directories 
    layout_dir = Path("output/pipeline_test_nc/layout")
    ocr_dir = Path("output/pipeline_test_nc/ocr")
 
    layout_dir.mkdir(parents=True, exist_ok=True)
    ocr_dir.mkdir(parents=True, exist_ok=True)   
  
    # --- filtering OCR results ---
    for res in output_ocr:
        # Save original OCR
        img_file = ocr_dir / f"{image_path.stem}_ocr.png"
        json_file = ocr_dir / f"{image_path.stem}_ocr.json"
        res.save_to_img(str(img_file))
        res.save_to_json(str(json_file))
            
    # --- Save Layout results ---
    for res in output_layout:
        img_file = layout_dir / f"{image_path.stem}_layout.png"
        json_file = layout_dir / f"{image_path.stem}_layout.json"
        res.save_to_img(str(img_file))
        res.save_to_json(str(json_file))
     
    print(f"Total processed in {time_after_ocr - now:.2f} seconds")
    print(f"Layout analysis: {time_after_layout - now:.2f} s")
    print(f"OCR: {time_after_ocr - time_after_layout:.2f} s")
 

In [9]:
run_image_pipeline_nc("../data/table1.jpg", layout_model, ocr_model)

The result has multiple img files need to be saved. But the `save_path` has been specified as `output\pipeline_test_nc\ocr\table1_ocr.png`!


Total processed in 4.13 seconds
Layout analysis: 2.32 s
OCR: 1.81 s


In [ ]:
run_image_pipeline_nc("data/sample_image4.png", layout_model, ocr_model)

The result has multiple img files need to be saved. But the `save_path` has been specified as `output\pipeline_test_nc\ocr\sample_image4_ocr.png`!


Total processed in 1.19 seconds
Layout analysis: 0.07 s
OCR: 1.12 s


### pp structurev3 end to end try out

In [ ]:
#!pip install "paddlex[ocr]==3.2.1"
#version should be the same as pip show paddlex

In [ ]:
from paddlex import create_pipeline
from pathlib import Path
import time 

output_path = Path("output/structureV3")
pipeline = create_pipeline(pipeline="PP-StructureV3")



Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\UVDoc`.
Creating model: ('PP-DocBlockLayout', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-DocBlockLayout`.
Creating model: ('PP-DocLayout_plus-L', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-DocLayout_plus-L`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet

In [ ]:
now = time.time()
output = pipeline.predict(
    input="data/sample_pdf2.pdf",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

output_path.mkdir(parents = True, exist_ok = True)
for res in output:
    res.print() ## Print the structured prediction output
    res.save_to_json(save_path=output_path) ## Save the structured JSON result of the current image
    res.save_to_markdown(save_path=output_path) ## Save the result of the current image in Markdown format
print(f"time taken: {(time.time() - now):.2f}")

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-OCRv5_server_rec`.
{'res': {'input_path': 'data/sample_pdf2.pdf', 'page_index': 0, 'model_settings': {'use_doc_preprocessor': False, 'use_seal_recognition': False, 'use_tabl

time taken: 41.89


### Layout Parsing model 

In [1]:
from paddlex import create_pipeline

pipeline_par = create_pipeline(pipeline="layout_parsing")

Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
D:\Fellowship\docuAgent\venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\UVDoc`.
Creating model: ('RT-DETR-H_layout_17cls', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\LOQ\.paddlex\official_models\RT-DETR-H_layout_17cls`.
Creating model: ('PP-OCRv4_server_det', None)
Model files already e

In [4]:
from pathlib import Path
import time 

output_path_parsing = Path("output/layout_parsing")
output_path_parsing.mkdir(parents = True, exist_ok = True)
now = time.time()    
output_par = pipeline_par.predict(
    input="data/sample_pdf2.pdf",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
for res in output_par:
    res.print()  ## Print the structured output of the prediction
    res.save_to_img(save_path="output/layout_parsing")  ## Save the visualized image results of all submodules for the current image
    res.save_to_json(save_path="output/layout_parsing")  ## Save the structured JSON results for the current image
    res.save_to_xlsx(save_path="output/layout_parsing")  ## Save the sub-table results in XLSX format for the current image
    res.save_to_html(save_path="output/layout_parsing")  ## Save the sub-table results in HTML format for the current image
print(f"time taken: {(time.time() - now):.2f}")

{'res': {'input_path': 'data/sample_pdf2.pdf', 'page_index': 0, 'model_settings': {'use_doc_preprocessor': False, 'use_seal_recognition': True, 'use_table_recognition': True, 'use_formula_recognition': False}, 'parsing_res_list': [{'block_bbox': [np.float32(84.89842), np.float32(52.73748), np.float32(874.09625), np.float32(141.63591)], 'block_label': 'image', 'block_content': 'eurostat\n0\neuroindicators'}, {'block_bbox': [np.float32(806.25793), np.float32(227.96178), np.float32(1115.4048), np.float32(253.70932)], 'block_label': 'text', 'block_content': '51/2022 － 29 April 2022'}, {'block_bbox': [np.float32(81.225945), np.float32(331.9846), np.float32(784.89105), np.float32(408.67432)], 'block_label': 'doc_title', 'block_content': 'Flash estimate - April 2022\nEuro area annual inflation up to 7.5%'}, {'block_bbox': [np.float32(81.225945), np.float32(331.9846), np.float32(784.89105), np.float32(408.67432)], 'block_label': 'text', 'block_content': 'Flash estimate - April 2022\nEuro area 

time taken: 17.72


In [5]:
output_path_parsing = Path("output/layout_parsing")
output_path_parsing.mkdir(parents = True, exist_ok = True)
now = time.time()    
output_par = pipeline_par.predict(
    input="data/sample_image1.png",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
for res in output_par:
    res.print()  ## Print the structured output of the prediction
    res.save_to_img(save_path="output/layout_parsing")  ## Save the visualized image results of all submodules for the current image
    res.save_to_json(save_path="output/layout_parsing")  ## Save the structured JSON results for the current image
    res.save_to_xlsx(save_path="output/layout_parsing")  ## Save the sub-table results in XLSX format for the current image
    res.save_to_html(save_path="output/layout_parsing")  ## Save the sub-table results in HTML format for the current image
print(f"time taken: {(time.time() - now):.2f}")

{'res': {'input_path': 'data/sample_image1.png', 'page_index': None, 'model_settings': {'use_doc_preprocessor': False, 'use_seal_recognition': True, 'use_table_recognition': True, 'use_formula_recognition': False}, 'parsing_res_list': [{'block_bbox': [np.float32(291.91058), np.float32(88.51835), np.float32(376.49576), np.float32(174.03365)], 'block_label': 'image', 'block_content': ''}, {'block_bbox': [np.float32(252.98378), np.float32(201.7795), np.float32(424.12827), np.float32(227.91924)], 'block_label': 'text', 'block_content': 'R&D QUALITY IMPROVEHENT\nSUGGESTION/SOLUTION FORM'}, {'block_bbox': [np.float32(89.21412), np.float32(268.7681), np.float32(593.93506), np.float32(286.26617)], 'block_label': 'text', 'block_content': 'Name/Phone Ext.: M.Hamann、P.Harper，P.Martinez\nDate:\n9/3/92'}, {'block_bbox': [np.float32(90.431915), np.float32(310.5399), np.float32(573.7966), np.float32(327.21423)], 'block_label': 'text', 'block_content': 'SupervLsor/Manager: J.S.Wigand\nR&D Group:Licens

time taken: 4.47


In [6]:
output_path_parsing = Path("output/layout_parsing")
output_path_parsing.mkdir(parents = True, exist_ok = True)
now = time.time()    
output_par = pipeline_par.predict(
    input="data/sample_image4.png",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
for res in output_par:
    res.print()  ## Print the structured output of the prediction
    res.save_to_img(save_path="output/layout_parsing")  ## Save the visualized image results of all submodules for the current image
    res.save_to_json(save_path="output/layout_parsing")  ## Save the structured JSON results for the current image
    res.save_to_xlsx(save_path="output/layout_parsing")  ## Save the sub-table results in XLSX format for the current image
    res.save_to_html(save_path="output/layout_parsing")  ## Save the sub-table results in HTML format for the current image
print(f"time taken: {(time.time() - now):.2f}")

{'res': {'input_path': 'data/sample_image4.png', 'page_index': None, 'model_settings': {'use_doc_preprocessor': False, 'use_seal_recognition': True, 'use_table_recognition': True, 'use_formula_recognition': False}, 'parsing_res_list': [{'block_bbox': [np.float32(8.560376), np.float32(55.331917), np.float32(642.8639), np.float32(739.1475)], 'block_label': 'table', 'block_content': '<html><body><table><tbody><tr><td colspan="6">WINSTON& STRAWN FaxCover Sheet</td></tr><tr><td colspan="2">MIAMI TRIAL SITE,3OOO FIRST UNION FINANCIAL CENTER,ZOO SOUTH BISCAYNE BOULEVARD,MIAMI,FLORIDA33I31 FAX:（305）400-6106 2OOPARK AVENUE</td><td>14OO L STREET. N,W 35 WEST WACKER WASHNGTON,DC2OCO5-3502 202-371-5700 312-558-5600 2SCAGO.L 60601</td><td colspan="2">751IG PANIS,FRANCE 2AvENUE VICrOR HUGO 33-1-53-64-82-82</td></tr><tr><td colspan="2">FROM: Kevin Narko</td><td colspan="2">FaxNumber:305400-6107</td><td colspan="2">CHARGEBACK: 4162/158</td></tr><tr><td colspan="6">DATE: 10/13/99</td></tr><tr><td colsp

time taken: 2.18


In [ ]:
import cv2
import numpy as np

def preprocess_region(img, x1, y1, x2, y2, pad_w=5, pad_h=5, min_height=32):
      # Add padding with boundary checks
    x1_pad = max(0, x1 - pad_w)
    y1_pad = max(0, y1 - pad_h)
    x2_pad = min(img.shape[1], x2 + pad_w)
    y2_pad = min(img.shape[0], y2 + pad_h)
    
    # Crop region
    region = img[y1_pad:y2_pad, x1_pad:x2_pad]
    
    # Upscale if region is too small
    h, w = region.shape[:2]
    if h < min_height:
        scale = min_height / h
        new_width = int(w * scale)
        region = cv2.resize(region, (new_width, min_height), interpolation=cv2.INTER_LINEAR)
    
    return region

In [ ]:
import os
import json
import time
import cv2
import numpy as np
from pathlib import Path
from paddleocr import FormulaRecognition

def run_image_pipeline_c(path, preprocessing_pipeline, layout_model, formula_model, ocr_model):
    image_path = Path(path)
    start_time = time.time()
    image_name = image_path.stem

    # --- Step 1: Layout detection ---
    layout_results = layout_model.predict(str(image_path))
    layout_time = time.time()

    # Create directories
    layout_dir = Path(f"output/pipeline_test/{image_name}/layout")
    ocr_dir = Path(f"output/pipeline_test/{image_name}/ocr")
    formula_dir = Path(f"output/pipeline_test/{image_name}/formula")
    for d in [layout_dir, ocr_dir, formula_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # layout model and its output
    for res in layout_results:
        layout_img = layout_dir / f"{image_name}_layout.png"
        layout_json = layout_dir / f"{image_name}_layout.json"
        res.save_to_img(str(layout_img))
        res.save_to_json(str(layout_json))

    # 
    img = cv2.imread(str(image_path))

    # Prepare OCR JSON output
    ocr_json_path = ocr_dir / f"{image_name}_ocr.json"
    if ocr_json_path.exists():
        with open(ocr_json_path, "r", encoding="utf-8") as f:
            all_ocr_results = json.load(f)
    else:
        all_ocr_results = {}

    SKIP_FOR_OCR = ["table", "figure_table_title", "chart", "image", "seal", "formula"]

    #loop through layout boxes
    for page_result in layout_results:
        for i, box in enumerate(page_result["boxes"]):
            cls = box['label']
            conf = box['score']
            coords = [int(x) for x in box['coordinate']]
            x1, y1, x2, y2 = coords
            
            region = preprocess_region(img, x1, y1, x2, y2)
            # --- OCR Regions preprocessed and cropped 
            if (cls not in SKIP_FOR_OCR):
                ocr_result = ocr_model.predict(region)
                out_img = ocr_dir / f"{image_name}_{cls}_{i}.png"
                cv2.imwrite(str(out_img), region)

                # Store OCR results per region
                if f"region_{i}" not in all_ocr_results:
                    all_ocr_results[f"region_{i}"] = []

                recognized_polys =[]
                for res in ocr_result:
                    structured = {}
                    structured["class"] = cls
                    structured["confidence"] = float(conf)
                    structured["bbox"] = [x1, y1, x2, y2]  # original region box
                    structured["recognized_texts"] = res['rec_texts']
                    structured["recognized_confidence"] = [float(score) for score in res['rec_scores']]
                    
                    for poly in res["rec_polys"]:
                         abs_poly = [[p[0] + x1, p[1] + y1] for p in poly]
                         recognized_polys.append(abs_poly)
                    structured["recognized_polys"] = [
                              poly.tolist() if isinstance(poly, np.ndarray) else poly
                                for poly in res['rec_polys']
                      ] 
                    all_ocr_results[f"region_{i}"]=structured

            # --- formula Regions evaluated by special model ---
            elif cls == "formula":
                formula_output = formula_model.predict(region)
                for idx, f_res in enumerate(formula_output):
                    formula_img_path = formula_dir / f"{image_name}_formula_{i}_{idx}.png"
                    formula_json_path = formula_dir / f"{image_name}_formula_{i}_{idx}.json"
                    f_res.save_to_img(save_path=str(formula_img_path))
                    f_res.save_to_json(save_path=str(formula_json_path))

    # Save OCR JSON
    with open(ocr_json_path, "w", encoding="utf-8") as f:
        json.dump(all_ocr_results, f, ensure_ascii=False, indent=4)

    end_time = time.time()
    print(f"Total Time: {end_time - start_time:.2f}s")
    print(f"Layout Detection Time: {layout_time - start_time:.2f}s")
    print(f"OCR + Formula Processing Time: {end_time - layout_time:.2f}s")


In [ ]:
run_image_pipeline_c(
    path="data/sample_image4.png",
    preprocessing_pipeline=None, 
    layout_model=layout_model,
    formula_model=formula_model,
    ocr_model=ocr_model
)

Total Time: 3.35s
Layout Detection Time: 0.16s
OCR + Formula Processing Time: 3.19s


In [ ]:
run_image_pipeline_c(
    path="data/page_1.png",
    preprocessing_pipeline=None,  
    layout_model=layout_model,
    formula_model=formula_model,
    ocr_model=ocr_model
)

Total Time: 1.80s
Layout Detection Time: 0.15s
OCR + Formula Processing Time: 1.66s


In [ ]:
import time, cv2, json, numpy as np, random
from pathlib import Path

from src.pipeline.image_preprocessing import preprocess_document

def image_pipeline_with_masking(
    img=None,
    img_path=None,
    pdf_name=None,
    preprocess=True,
    layout_result_save=False,
    pdf_page_num=1,
    ocr_img_save=False,
    mask_padding=10  # <-- NEW: padding in pixels around masked regions
):
    
    if img is None and img_path is None:
        raise ValueError("Either img or img_path must be provided.")
    
    # --- Load image if path provided ---
    if img is None and img_path is not None:
        img = cv2.imread(str(img_path))
        print("[INFO] Image loaded from path:", img_path)
    
    # --- Naming setup ---
    if img_path:
        img_name = Path(img_path).stem
    elif pdf_name:
        img_name = f"{pdf_name}_page_{pdf_page_num}"
    else:
        img_name = str(random.randint(1000, 9999))
        
    file_name = pdf_name if pdf_name else img_name  # for output dirs
    SKIP_FOR_OCR = ["table", "chart", "image", "seal", "formula"]

    # --- Directories ---
    OUTPUT = Path("output")
    OUTPUT.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT / file_name
    output_path.mkdir(parents=True, exist_ok=True)

    print(f"\n[INFO] Starting pipeline for: {img_name}")

    total_start = time.time()

    # --- Preprocessing ---
    if preprocess:
        print(f"[INFO] Preprocessing enabled for {img_name}")
        preprocess_start = time.time()
        # preprocess pipeline not implemented so 
        preprocess_document(img) 
        # img = preprocess_document(img)  
        preprocess_end = time.time()
        print(f"[TIME] Preprocessing took {preprocess_end - preprocess_start:.2f} seconds")

    # --- Layout Detection ---
    layout_start = time.time()
    layout_results = layout_model.predict(img)
    layout_end = time.time()
    print(f"[TIME] Layout detection took {layout_end - layout_start:.2f} seconds")

    if layout_result_save:
        layout_dir = output_path / "layout"
        layout_dir.mkdir(parents=True, exist_ok=True)
        for res in layout_results:
            res.save_to_json(str(layout_dir / f"{img_name}_layout.json"))
            res.save_to_img(str(layout_dir / f"{img_name}_layout.png"))

    # --- Masking Stage ---

    cv2.imwrite(str(output_path / f"{img_name}_masked.png"), masked_image_for_ocr)

    # --- OCR Stage ---
    ocr_start = time.time()
    ocr_results = ocr_model.predict(masked_image_for_ocr)
    ocr_end = time.time()
    print(f"[TIME] OCR took {ocr_end - ocr_start:.2f} seconds")

    # --- Save OCR Results ---
    
    for res in layout_results:
        res.save_to_img(str(output_path))
        res.save_to_json(str(output_path))
    for res in ocr_results:
        res.save_to_img(str(output_path))
        res.save_to_json(str(output_path))

    total_end = time.time()
    print(f"[TOTAL TIME] Pipeline for {img_name} took {total_end - total_start:.2f} seconds\n")

    return {
        "layout_time": layout_end - layout_start,
        "mask_time": masking_end - masking_start,
        "ocr_time": ocr_end - ocr_start,
        "total_time": total_end - total_start
    }


In [34]:
image_pipeline(img_path = "data/page_1.png")

[INFO] Image loaded from path: data/page_1.png

[INFO] Starting pipeline for: page_1
[INFO] Preprocessing enabled for page_1
[TIME] Preprocessing took 0.00 seconds
[TIME] Layout detection took 0.08 seconds
[TIME] Masking took 0.00 seconds


There is not input file name as reference for name of saved result file. So the saved result file would be named with timestamp and random number: `1760893239_1531`.


[TIME] OCR took 6.25 seconds


There is not input file name as reference for name of saved result file. So the saved result file would be named with timestamp and random number: `1760893239_4130`.


[TOTAL TIME] Pipeline for page_1 took 7.36 seconds



{'layout_time': 0.07614254951477051,
 'mask_time': 0.004439353942871094,
 'ocr_time': 6.2483179569244385,
 'total_time': 7.357322454452515}

In [35]:
image_pipeline(img_path = "data/sample_image4.png")

[INFO] Image loaded from path: data/sample_image4.png

[INFO] Starting pipeline for: sample_image4
[INFO] Preprocessing enabled for sample_image4
[TIME] Preprocessing took 0.00 seconds
[TIME] Layout detection took 0.07 seconds
[TIME] Masking took 0.00 seconds


There is not input file name as reference for name of saved result file. So the saved result file would be named with timestamp and random number: `1760893240_4632`.
There is not input file name as reference for name of saved result file. So the saved result file would be named with timestamp and random number: `1760893241_2146`.


[TIME] OCR took 0.60 seconds
[TOTAL TIME] Pipeline for sample_image4 took 0.88 seconds



{'layout_time': 0.07334518432617188,
 'mask_time': 0.0004055500030517578,
 'ocr_time': 0.6021363735198975,
 'total_time': 0.8806145191192627}

In [36]:
ocr_beg = time.time()
ocr_results = ocr_model.predict("data/page_1.png")
for res in ocr_results:
    res.save_to_img("output")
    res.save_to_json("output")
ocr_end = time.time()

print(f"The time taken is {ocr_end - ocr_beg:.2f}")

The time taken is 10.42


In [38]:
image_pipeline(img_path ="data/sample_image5.png")

[INFO] Image loaded from path: data/sample_image5.png

[INFO] Starting pipeline for: sample_image5
[INFO] Preprocessing enabled for sample_image5
[TIME] Preprocessing took 0.00 seconds
[TIME] Layout detection took 0.09 seconds
[TIME] Masking took 0.00 seconds


There is not input file name as reference for name of saved result file. So the saved result file would be named with timestamp and random number: `1760893801_1531`.
There is not input file name as reference for name of saved result file. So the saved result file would be named with timestamp and random number: `1760893801_9549`.


[TIME] OCR took 0.41 seconds
[TOTAL TIME] Pipeline for sample_image5 took 0.67 seconds



{'layout_time': 0.08731436729431152,
 'mask_time': 0.0004608631134033203,
 'ocr_time': 0.4055187702178955,
 'total_time': 0.6689748764038086}

In [ ]:
import time, cv2, random
from pathlib import Path
from paths import OUTPUT

from src.pipeline.layout_on_image import layout_on_image
from src.pipeline.image_preprocessing import preprocess_document
from src.pipeline.ocr_on_image import ocr_on_image
from src.pipeline.masking_on_image import mask_regions
from src.utils.save_results import save_results

def image_pipeline_with_masking(
    img=None,
    img_path=None,
    pdf_name=None,
    preprocess=True,
    layout_result_save=False,
   
    pdf_page_num=1,

):
    if img is None and img_path is None:
        raise ValueError("Either img or img_path must be provided.")

    # --- load image if path is provided ---
    if img is None:
        img = cv2.imread(str(img_path))
        print("[INFO] Image loaded:", img_path)

    # --- Output naming ---
    if img_path:
        img_name = Path(img_path).stem
    elif pdf_name:
        img_name = f"{pdf_name}_page_{pdf_page_num}"
    else:
        img_name = str(random.randint(1000, 9999))
 
    file_name = pdf_name if pdf_name else img_name  # for output dirs

    print(f"\n[INFO] Starting image pipeline for: {img_name}")
    image_start = time.time()

    # --- Step 1: Preprocess ---
    if preprocess:
        
        preprocess_document(img)


    # --- Step 2: Layout Detection ---
    layout_results = layout_on_image(img, img_name)
 
    # --- Step 3: Masking ---
    masked_img = mask_regions(img, img_name, layout_results)
    # --- Step 4: OCR ---
    ocr_results = ocr_on_image(masked_img, img_name)
    
    structured_ocr_results = {}
    for res in ocr_results:
        structured_ocr_results["rec_polys"] = [
            poly.tolist() if isinstance(poly, np.ndarray) else poly
            for poly in res['rec_polys']
            ]
        structured_ocr_results["rec_scores"] = [float(score) for score in res['rec_scores']]
        structured_ocr_results["rec_texts"] = res['rec_texts']

    if layout_result_save:
        save_results(OUTPUT, file_name,img_name, img, masked_img, page_num = pdf_page_num, layout_results=layout_results, layout_result_save=layout_result_save, ocr_results=structured_ocr_results)
    else:
        save_results(OUTPUT, file_name,img_name, img, masked_img, page_num = pdf_page_num, layout_results=None, layout_result_save=layout_result_save, ocr_results=structured_ocr_results)
    image_end = time.time()
    print(f"[TOTAL TIME] {img_name}: {image_end - image_start:.2f}s\n")




In [ ]:
def mask_regions(image,image_name, layout_results):
    
    start_time = time.time()
    masked_img = image.copy()
    mask_padding=10
    
    for page_result in layout_results:
        for box in page_result["boxes"]:
            cls = box['label']
            coords = [int(x) for x in box['coordinate']]
            x1, y1, x2, y2 = coords
            
            SKIP_FOR_OCR = ["table", "chart", "image", "seal", "formula"]
            if cls in SKIP_FOR_OCR:
                # Apply padding
                x1_pad = max(0, x1 - mask_padding)
                y1_pad = max(0, y1 - mask_padding)
                x2_pad = min(image.shape[1], x2 + mask_padding)
                y2_pad = min(image.shape[0], y2 + mask_padding)
                
                # Mask region
                cv2.rectangle(masked_img, (x1_pad, y1_pad), (x2_pad, y2_pad), (255, 255, 255), thickness=-1)
                
    end_time = time.time()
    print(f"[INFO] Masking Time for {image_name} : {end_time - start_time:.2f}s")
    return masked_img

In [ ]:
import cv2, json

def save_results(OUTPUT,file_name,img_name, img, page_num, masked_img,ocr_results, layout_results=None,layout_results_save=False):
    
    output_path = OUTPUT / file_name
    output_path.mkdir(parents=True, exist_ok=True)
    
    preprocessed_img_path = output_path / f"{img_name}_preprocessed.png"
    masked_img_path = output_path / f"{img_name}_masked.png"
    
    cv2.imwrite(str(preprocessed_img_path), img)
    cv2.imwrite(str(masked_img_path), masked_img)
    
    # Save layout results
    layout_dir = output_path / "layout"
    layout_dir.mkdir(parents=True, exist_ok=True)

    layout_img_path = layout_dir / f"{img_name}_layout.png"
    layout_json_path = layout_dir / f"{img_name}_layout.json"
    
    for res in layout_results:
        res.save_to_json(str(layout_json_path))
        res.save_to_img(str(layout_img_path))



    # Save OCR results
    ocr_dir = output_path / "ocr"
    ocr_dir.mkdir(parents=True, exist_ok=True)
    
    ocr_json_path = ocr_dir / f"{file_name}_ocr.json"
    ocr_img_path = ocr_dir / f"{img_name}_ocr.png"

    # Load existing JSON if it exists
    if ocr_json_path.exists():
        with open(ocr_json_path, "r", encoding="utf-8") as f:
            try:
                all_ocr_results = json.load(f)
            except json.JSONDecodeError:
                all_ocr_results = {}
    else:
        all_ocr_results = {}

    # Add or update the page
    all_ocr_results[f"page_{page_num}"] = ocr_results

    # Write back to file
    with open(ocr_json_path, "w", encoding="utf-8") as f:
        json.dump(all_ocr_results, f, indent=4, ensure_ascii=False)

    print(f"[INFO] OCR results saved for page {page_num} → {ocr_json_path}")
